# Stage D: Cross-Sensor Identity Linking & Direction

See `docs/specs/2026-07-30-stage-d-direction-linking-design.md` for the
full design. This notebook: (1) reproduces the three ground-truth
findings that motivated the design, (2) calibrates the matcher against
`DATA/combined_file_manual.csv`, (3) applies it to all dual-sensor dates,
(4) writes the matched-pair output table.

In [1]:
import sys
sys.path.insert(0, ".")
import pandas as pd
import numpy as np
from stage_d_lib import (
    first_session_anchor, last_session_anchor, best_direction_for_pair,
    match_travelers, predicted_counts, calibration_loss, manual_to_internal,
    date_predictions, grid_search, grid_search_orientation, leave_one_date_out,
    DEFAULT_MAX_TRANSIT_S, RSSI_TREND_SCALE,
)

DATA_DIR = "../DATA"

## Load Stage C output and the manual ground truth

In [2]:
sess_a = pd.read_csv(f"{DATA_DIR}/sessions_sensor_A_with_traveler.csv")
sess_b = pd.read_csv(f"{DATA_DIR}/sessions_sensor_B_with_traveler.csv")
for df in (sess_a, sess_b):
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])

manual = pd.read_csv(f"{DATA_DIR}/combined_file_manual.csv")
manual["dt"] = pd.to_datetime(manual["Date"] + " " + manual["Time"], format="%d.%m.%Y %H:%M:%S")
manual["date"] = manual["dt"].dt.date
gt_windows = manual.groupby("date").agg(window_start=("dt", "min"), window_end=("dt", "max")).reset_index()
gt_dirs = manual.groupby(["date", "direction"]).size().unstack(fill_value=0).reset_index()
gt_windows = gt_windows.merge(gt_dirs, on="date")
print(f"Sensor A sessions: {len(sess_a)}, Sensor B sessions: {len(sess_b)}")
print(f"GT dates: {len(gt_windows)}")
gt_windows

Sensor A sessions: 15267, Sensor B sessions: 19359
GT dates: 8


,date,window_start,window_end,A,B
0,2026-04-11,2026-04-11 09:03:54,2026-04-11 09:36:57,1,27
1,2026-05-02,2026-05-02 12:36:26,2026-05-02 13:15:03,36,20
2,2026-05-09,2026-05-09 11:00:35,2026-05-09 11:58:39,56,181
3,2026-05-16,2026-05-16 08:20:51,2026-05-16 09:04:17,2,22
4,2026-05-19,2026-05-19 09:27:20,2026-05-19 10:43:42,2,136
5,2026-05-23,2026-05-23 09:04:50,2026-05-23 09:57:14,13,30
6,2026-05-27,2026-05-27 10:29:41,2026-05-27 10:51:16,0,17
7,2026-05-31,2026-05-31 10:09:30,2026-05-31 10:51:04,1,55


## Ground-truth finding 1: sensors see mostly the same people

Summing each sensor's independent traveler count double-counts, because
most travelers show up on both sensors within the same short window.

In [3]:
def travelers_in_window(sess_df, ws, we):
    win = sess_df[(sess_df["start_time"] >= ws) & (sess_df["start_time"] <= we)]
    return win.groupby("Traveler_ID")["MAC"].apply(set)

overlap_rows = []
for _, r in gt_windows.iterrows():
    ta = travelers_in_window(sess_a, r["window_start"], r["window_end"])
    tb = travelers_in_window(sess_b, r["window_start"], r["window_end"])
    shared_macs = set().union(*ta.values) & set().union(*tb.values) if len(ta) and len(tb) else set()
    overlap_rows.append({
        "date": r["date"], "trav_A": len(ta), "trav_B": len(tb),
        "manual_total": int(r["A"] + r["B"]),
    })
overlap_df = pd.DataFrame(overlap_rows)
print("Sum of A+B traveler counts vs. manual total (should NOT match if double-counting):")
overlap_df

Sum of A+B traveler counts vs. manual total (should NOT match if double-counting):


,date,trav_A,trav_B,manual_total
0,2026-04-11,42,42,28
1,2026-05-02,138,75,56
2,2026-05-09,265,268,237
3,2026-05-16,46,49,24
4,2026-05-19,124,140,138
5,2026-05-23,53,56,43
6,2026-05-27,22,22,17
7,2026-05-31,33,25,56


## Ground-truth finding 2: raw MAC-address matching over-counts

Matching Sensor A and B by literal MAC equality (no per-sensor entity
resolution first) produces a much larger union than the manual total.

In [4]:
a_raw = pd.read_parquet(f"{DATA_DIR}/sensor_A_deduped.parquet")[["MAC", "timestamp"]]
b1 = pd.read_excel(f"{DATA_DIR}/combined_file_sensor_B_part_1.xlsx")
b2 = pd.read_excel(f"{DATA_DIR}/combined_file_sensor_B_part_2.xlsx")
b_raw = pd.concat([b1, b2], ignore_index=True).drop_duplicates(subset=["MAC", "RSSI", "timestamp"])[["MAC", "timestamp"]]

raw_union_rows = []
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    a_macs = set(a_raw[(a_raw["timestamp"] >= ws) & (a_raw["timestamp"] <= we)]["MAC"])
    b_macs = set(b_raw[(b_raw["timestamp"] >= ws) & (b_raw["timestamp"] <= we)]["MAC"])
    union_total = len(a_macs | b_macs)
    manual_total = int(r["A"] + r["B"])
    raw_union_rows.append({
        "date": r["date"], "raw_mac_union": union_total, "manual_total": manual_total,
        "ratio": round(union_total / manual_total, 2) if manual_total else None,
    })
pd.DataFrame(raw_union_rows)

,date,raw_mac_union,manual_total,ratio
0,2026-04-11,64,28,2.29
1,2026-05-02,247,56,4.41
2,2026-05-09,590,237,2.49
3,2026-05-16,64,24,2.67
4,2026-05-19,370,138,2.68
5,2026-05-23,90,43,2.09
6,2026-05-27,37,17,2.18
7,2026-05-31,52,56,0.93


## Ground-truth finding 3: naive "first-seen" order is not a direction signal

Comparing which sensor first detected a shared MAC, against the manual
direction split, shows the opposite skew on some dates -- consistent with
Sensor A having a longer detection range, not true arrival order.

In [5]:
first_seen_rows = []
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    a_first = a_raw[(a_raw["timestamp"] >= ws) & (a_raw["timestamp"] <= we)].groupby("MAC")["timestamp"].min()
    b_first = b_raw[(b_raw["timestamp"] >= ws) & (b_raw["timestamp"] <= we)].groupby("MAC")["timestamp"].min()
    shared = set(a_first.index) & set(b_first.index)
    a_seen_first = sum(1 for m in shared if a_first[m] < b_first[m])
    b_seen_first = sum(1 for m in shared if b_first[m] < a_first[m])
    first_seen_rows.append({
        "date": r["date"], "A_seen_first": a_seen_first, "B_seen_first": b_seen_first,
        "manual_A": int(r["A"]), "manual_B": int(r["B"]),
    })
pd.DataFrame(first_seen_rows)

,date,A_seen_first,B_seen_first,manual_A,manual_B
0,2026-04-11,52,1,1,27
1,2026-05-02,39,60,36,20
2,2026-05-09,341,113,56,181
3,2026-05-16,47,5,2,22
4,2026-05-19,223,26,2,136
5,2026-05-23,38,28,13,30
6,2026-05-27,35,0,0,17
7,2026-05-31,29,4,1,55


## Build per-traveler boundary anchors for each GT window

In [6]:
def build_anchors_for_window(sess_df, ws, we):
    win = sess_df[(sess_df["start_time"] >= ws) & (sess_df["start_time"] <= we)]
    anchors = []
    for tid, grp in win.groupby("Traveler_ID"):
        anchors.append({
            "traveler_id": tid,
            "first": first_session_anchor(grp),
            "last": last_session_anchor(grp),
        })
    return anchors

gt_dates_data = {}
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    gt_dates_data[str(r["date"])] = {
        "a_anchors": build_anchors_for_window(sess_a, ws, we),
        "b_anchors": build_anchors_for_window(sess_b, ws, we),
        "manual_A": int(r["A"]), "manual_B": int(r["B"]),
    }

for date, d in gt_dates_data.items():
    print(f"{date}: {len(d['a_anchors'])} A travelers, {len(d['b_anchors'])} B travelers, "
          f"manual A={d['manual_A']} B={d['manual_B']}")

2026-04-11: 42 A travelers, 42 B travelers, manual A=1 B=27
2026-05-02: 138 A travelers, 75 B travelers, manual A=36 B=20
2026-05-09: 265 A travelers, 268 B travelers, manual A=56 B=181
2026-05-16: 46 A travelers, 49 B travelers, manual A=2 B=22
2026-05-19: 124 A travelers, 140 B travelers, manual A=2 B=136
2026-05-23: 53 A travelers, 56 B travelers, manual A=13 B=30
2026-05-27: 22 A travelers, 22 B travelers, manual A=0 B=17
2026-05-31: 33 A travelers, 25 B travelers, manual A=1 B=55


### Structural ceiling: can the matcher even reach the manual totals?

A bipartite matching can produce at most `min(n_A, n_B)` pairs. If that
ceiling is below the manual ground-truth total for a date, **no** parameter
setting can hit that date's target, and the squared-error loss will reward
"match as many people as possible" on that date no matter what. Worth
checking before reading anything into the calibrated parameters.

In [7]:
cap_rows = []
for date, d in gt_dates_data.items():
    cap = min(len(d["a_anchors"]), len(d["b_anchors"]))
    total = d["manual_A"] + d["manual_B"]
    cap_rows.append({"date": date, "n_A": len(d["a_anchors"]), "n_B": len(d["b_anchors"]),
                     "max_possible_matches": cap, "manual_total": total,
                     "reachable": cap >= total})
cap_df = pd.DataFrame(cap_rows)
print(f"{(~cap_df['reachable']).sum()} of {len(cap_df)} GT dates have a manual total "
      f"the matcher cannot reach even by pairing every available traveler.")
cap_df

2 of 8 GT dates have a manual total the matcher cannot reach even by pairing every available traveler.


,date,n_A,n_B,max_possible_matches,manual_total,reachable
0,2026-04-11,42,42,42,28,True
1,2026-05-02,138,75,75,56,True
2,2026-05-09,265,268,265,237,True
3,2026-05-16,46,49,46,24,True
4,2026-05-19,124,140,124,138,False
5,2026-05-23,53,56,53,43,True
6,2026-05-27,22,22,22,17,True
7,2026-05-31,33,25,25,56,False


## Calibration: leave-one-date-out fit (params AND label orientation)

Free parameters (design doc §5): the transit-time window's centre/spread,
the `rssi_shape_score` weight relative to timing, and the minimum
match-score threshold. `max_transit_s` is *not* fitted — it is a hard
physical gate (see below), not a tuning knob.

Two things are fitted jointly, **both inside every CV fold**:

1. the four grid parameters, and
2. the label orientation `flip` — the manual CSV's A/B convention is
   undocumented, so which physical direction the observer's "A" means is
   itself an inference from the ground truth.

Choosing the orientation once over all 8 dates and then reusing it inside
each fold would leak the held-out date's own ground truth into its
evaluation, so `leave_one_date_out` re-fits it per fold from that fold's
7 training dates only.

### Score scale

`match_score = transit_time_score + rssi_weight * rssi_shape_score`, where
both terms are bounded to [0, 1]. So the maximum attainable score is
`1.0 + rssi_weight`, and `min_match_score` must be read against that.
Both terms are 0 for any candidate whose transit gap is negative or longer
than `max_transit_s` (300s), so such a pair can never be matched.

In [8]:
import itertools

# Grid ranges are sized for the *bounded* [0,1] rssi_shape_score. The
# original grid (rssi_weight up to 5.0, min_match_score up to 0.6) was
# tuned against the old unbounded shape score, whose typical magnitude was
# an order of magnitude larger; carrying those ranges over would make the
# RSSI term dominate again. Max attainable score is 1.0 + rssi_weight.
#
# These ranges were then widened once, after a first fit put
# transit_spread_s, rssi_weight and min_match_score all on a grid edge:
# spread now reaches 60s, rssi_weight reaches down to 0.0 (an explicit
# "does the RSSI shape term help at all?" hypothesis, not a filler value)
# and min_match_score down to 0.05. Note min_match_score must stay > 0 --
# an out-of-gate candidate scores exactly 0 and is rejected only because
# the threshold is strictly positive.
PARAM_GRID = [
    {"transit_center_s": c, "transit_spread_s": s, "rssi_weight": w,
     "min_match_score": m, "max_transit_s": DEFAULT_MAX_TRANSIT_S}
    for c, s, w, m in itertools.product(
        [10.0, 15.0, 20.0, 25.0, 30.0, 40.0, 50.0],
        [8.0, 12.0, 20.0, 30.0, 45.0, 60.0],
        [0.0, 0.1, 0.25, 0.5, 1.0, 2.0],
        [0.05, 0.15, 0.3, 0.5, 0.8, 1.1],
    )
]
print(f"Grid size: {len(PARAM_GRID)}")
print(f"rssi_shape_score saturation scale: {RSSI_TREND_SCALE} dB/s; "
      f"max transit gate: {DEFAULT_MAX_TRANSIT_S:.0f}s")

# Matching is independent of the label orientation, so every (date, params)
# match is computed exactly once and reused across both orientations and
# all 8 CV folds.
PREDS = date_predictions(gt_dates_data, PARAM_GRID)

for flip in (False, True):
    _, loss = grid_search(gt_dates_data, PARAM_GRID, flip, preds=PREDS)
    print(f"in-sample best total loss with flip={flip}: {loss:.1f}")

Grid size: 1512
rssi_shape_score saturation scale: 0.5 dB/s; max transit gate: 300s


in-sample best total loss with flip=False: 23146.0
in-sample best total loss with flip=True: 12412.0


In [9]:
loo_rows = leave_one_date_out(gt_dates_data, PARAM_GRID, preds=PREDS)
loo_report = pd.DataFrame([
    {
        "date": r["date"], "fold_flip": r["flip"],
        "predicted_A_to_B": r["predicted"]["A_to_B"], "predicted_B_to_A": r["predicted"]["B_to_A"],
        "predicted_total": r["predicted"]["total"],
        "manual_A_to_B": r["manual"]["A_to_B"], "manual_B_to_A": r["manual"]["B_to_A"],
        "manual_total": r["manual"]["total"],
        "held_out_loss": r["held_out_loss"],
    }
    for r in loo_rows
])
print("Leave-one-date-out held-out predictions vs. ground truth")
print("(params and orientation fitted on the other 7 dates only):")
for r in loo_rows:
    print(f"  {r['date']}: flip={r['flip']} params={r['params']}")
loo_report

Leave-one-date-out held-out predictions vs. ground truth
(params and orientation fitted on the other 7 dates only):
  2026-04-11: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 60.0, 'rssi_weight': 0.0, 'min_match_score': 0.05, 'max_transit_s': 300.0}
  2026-05-02: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 60.0, 'rssi_weight': 0.0, 'min_match_score': 0.05, 'max_transit_s': 300.0}
  2026-05-09: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 45.0, 'rssi_weight': 0.0, 'min_match_score': 0.3, 'max_transit_s': 300.0}
  2026-05-16: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 60.0, 'rssi_weight': 0.0, 'min_match_score': 0.05, 'max_transit_s': 300.0}
  2026-05-19: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 30.0, 'rssi_weight': 0.25, 'min_match_score': 0.05, 'max_transit_s': 300.0}
  2026-05-23: flip=True params={'transit_center_s': 10.0, 'transit_spread_s': 60.0, 'rssi_weight': 0.0, 'min_match_scor

,date,fold_flip,predicted_A_to_B,predicted_B_to_A,predicted_total,manual_A_to_B,manual_B_to_A,manual_total,held_out_loss
0,2026-04-11,True,30,7,37,27,1,28,126
1,2026-05-02,True,36,39,75,20,36,56,626
2,2026-05-09,True,148,110,258,181,56,237,4446
3,2026-05-16,True,25,17,42,22,2,24,558
4,2026-05-19,True,61,48,109,136,2,138,8582
5,2026-05-23,True,18,29,47,30,13,43,416
6,2026-05-27,True,11,10,21,17,0,17,152
7,2026-05-31,True,15,8,23,55,1,56,2738


In [10]:
final_params, best_orientation, final_loss = grid_search_orientation(
    gt_dates_data, PARAM_GRID, preds=PREDS)
print(f"Final production params (fit on all 8 GT dates): {final_params}")
print(f"Chosen label orientation: flip={best_orientation}")
print(f"In-sample total loss: {final_loss:.1f}  |  mean held-out loss: {loo_report['held_out_loss'].mean():.1f}")

# Boundary check: a fitted value sitting at the edge of its searched range
# means the true optimum is probably outside the grid.
for key in ("transit_center_s", "transit_spread_s", "rssi_weight", "min_match_score"):
    vals = sorted({p[key] for p in PARAM_GRID})
    v = final_params[key]
    edge = " <-- ON GRID BOUNDARY" if v in (vals[0], vals[-1]) else ""
    print(f"  {key}: {v} in {vals}{edge}")

Final production params (fit on all 8 GT dates): {'transit_center_s': 10.0, 'transit_spread_s': 60.0, 'rssi_weight': 0.0, 'min_match_score': 0.05, 'max_transit_s': 300.0}
Chosen label orientation: flip=True
In-sample total loss: 12412.0  |  mean held-out loss: 2205.5
  transit_center_s: 10.0 in [10.0, 15.0, 20.0, 25.0, 30.0, 40.0, 50.0] <-- ON GRID BOUNDARY
  transit_spread_s: 60.0 in [8.0, 12.0, 20.0, 30.0, 45.0, 60.0] <-- ON GRID BOUNDARY
  rssi_weight: 0.0 in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0] <-- ON GRID BOUNDARY
  min_match_score: 0.05 in [0.05, 0.15, 0.3, 0.5, 0.8, 1.1] <-- ON GRID BOUNDARY


### Is a boundary hit a truncated grid, or a saturation plateau?

A fitted value on the edge of its searched range usually means the true
optimum lies outside the grid. Not necessarily here: the loosest corner of
this grid (small centre, huge spread, zero RSSI weight, near-zero
threshold) makes the score nearly constant across the whole
`(0, max_transit_s]` gate, so the matcher saturates at "pair everyone you
can within the gate" and *cannot* go further. The check below evaluates
several explicitly out-of-grid parameter sets: if their loss is no better
than the fitted corner's, the boundary is a plateau, not a truncation, and
widening the grid again would buy nothing.

In [11]:
beyond_grid = [
    ("fitted optimum (grid corner)", final_params),
    ("centre 5s, spread 150s", {**final_params, "transit_center_s": 5.0, "transit_spread_s": 150.0}),
    ("centre 1s, spread 1000s, min 0.001", {**final_params, "transit_center_s": 1.0,
                                            "transit_spread_s": 1000.0, "min_match_score": 0.001}),
    ("flat limit (spread 1e6, min 1e-9)", {**final_params, "transit_center_s": 0.0,
                                           "transit_spread_s": 1e6, "min_match_score": 1e-9}),
]
probe_rows = []
for label, p in beyond_grid:
    loss, totals = 0.0, []
    for date, d in gt_dates_data.items():
        pr = predicted_counts(match_travelers(d["a_anchors"], d["b_anchors"], p))
        loss += calibration_loss(pr, manual_to_internal(d["manual_A"], d["manual_B"], best_orientation))
        totals.append(pr["total"])
    probe_rows.append({"params": label, "in_sample_loss": loss, "per_date_totals": totals})
probe_df = pd.DataFrame(probe_rows)
print("If no out-of-grid row beats the fitted corner, the boundary is a plateau:")
probe_df

If no out-of-grid row beats the fitted corner, the boundary is a plateau:


,params,in_sample_loss,per_date_totals
0,fitted optimum (grid corner),12412.0,"[37, 75, 265, 42, 110, 47, 21, 23]"
1,"centre 5s, spread 150s",13412.0,"[38, 75, 265, 43, 113, 51, 21, 25]"
2,"centre 1s, spread 1000s, min 0.001",13466.0,"[38, 75, 265, 43, 113, 51, 21, 25]"
3,"flat limit (spread 1e6, min 1e-9)",13694.0,"[38, 75, 265, 43, 113, 51, 21, 25]"


### Does the direction split actually track ground truth?

The aggregate loss can look acceptable while the *direction* column is
worthless (a near-50/50 prediction on a date whose true traffic is
overwhelmingly one-way still gets the total right). This table checks the
directional skew explicitly: predicted A→B share vs. manual A→B share,
per GT date, using the held-out (leave-one-date-out) predictions.

In [12]:
skew = loo_report.copy()
skew["pred_A_share"] = (skew["predicted_A_to_B"] /
                        skew["predicted_total"].replace(0, np.nan)).round(3)
skew["manual_A_share"] = (skew["manual_A_to_B"] / skew["manual_total"]).round(3)
skew["share_error"] = (skew["pred_A_share"] - skew["manual_A_share"]).round(3)
skew_report = skew[["date", "fold_flip", "predicted_A_to_B", "predicted_B_to_A",
                    "manual_A_to_B", "manual_B_to_A",
                    "pred_A_share", "manual_A_share", "share_error"]]
mean_abs_err = skew_report["share_error"].abs().mean()
# Baseline: a model that always predicts a 50/50 split, i.e. no direction
# signal whatsoever. If our error is not clearly below this, the direction
# column is not carrying information.
coin_flip_err = (0.5 - skew["manual_A_share"]).abs().mean()
corr = skew["pred_A_share"].corr(skew["manual_A_share"])
print(f"Mean |share error|:            {mean_abs_err:.3f}")
print(f"Always-50/50 baseline error:   {coin_flip_err:.3f}")
print(f"corr(predicted share, manual share) across the 8 dates: {corr:.3f}")
print("(0.0 error = direction perfectly recovered. An error at or above the "
      "baseline, or a correlation near/below 0, means the direction column "
      "is a coin flip and should not be trusted downstream.)")
skew_report

Mean |share error|:            0.292
Always-50/50 baseline error:   0.369
corr(predicted share, manual share) across the 8 dates: 0.533
(0.0 error = direction perfectly recovered. An error at or above the baseline, or a correlation near/below 0, means the direction column is a coin flip and should not be trusted downstream.)


,date,fold_flip,predicted_A_to_B,predicted_B_to_A,manual_A_to_B,manual_B_to_A,pred_A_share,manual_A_share,share_error
0,2026-04-11,True,30,7,27,1,0.811,0.964,-0.153
1,2026-05-02,True,36,39,20,36,0.480,0.357,0.123
2,2026-05-09,True,148,110,181,56,0.574,0.764,-0.190
3,2026-05-16,True,25,17,22,2,0.595,0.917,-0.322
4,2026-05-19,True,61,48,136,2,0.560,0.986,-0.426
5,2026-05-23,True,18,29,30,13,0.383,0.698,-0.315
6,2026-05-27,True,11,10,17,0,0.524,1.000,-0.476
7,2026-05-31,True,15,8,55,1,0.652,0.982,-0.330


## Apply to all 14 dual-sensor dates

Not just the 8 GT dates -- the calibrated matcher is applied to every
date where both sensors have data.

In [13]:
a_dates = set(sess_a["start_time"].dt.date)
b_dates = set(sess_b["start_time"].dt.date)
dual_dates = sorted(a_dates & b_dates)
print(f"{len(dual_dates)} dual-sensor dates: {dual_dates}")

all_matches = []
unmatched_rows = []
for date in dual_dates:
    ws = pd.Timestamp(date)
    we = ws + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    a_anchors = build_anchors_for_window(sess_a, ws, we)
    b_anchors = build_anchors_for_window(sess_b, ws, we)
    matches = match_travelers(a_anchors, b_anchors, final_params)
    for m in matches:
        m["date"] = date
    all_matches.extend(matches)

    matched_a = {m["traveler_id_a"] for m in matches}
    matched_b = {m["traveler_id_b"] for m in matches}
    for a in a_anchors:
        if a["traveler_id"] not in matched_a:
            unmatched_rows.append({"date": date, "sensor": "A", "traveler_id": a["traveler_id"]})
    for b in b_anchors:
        if b["traveler_id"] not in matched_b:
            unmatched_rows.append({"date": date, "sensor": "B", "traveler_id": b["traveler_id"]})

matched_df = pd.DataFrame(all_matches)
unmatched_df = pd.DataFrame(unmatched_rows)
print(f"Total matched pairs across all dual-sensor dates: {len(matched_df)}")
print(f"Total unmatched (single-sensor-only) travelers: {len(unmatched_df)}")
print()
print("Per-date direction split of the matched pairs:")
per_date = matched_df.groupby(["date", "direction"]).size().unstack(fill_value=0)
per_date["total"] = per_date.sum(axis=1)
if "A_to_B" in per_date:
    per_date["A_to_B_share"] = (per_date["A_to_B"] / per_date["total"]).round(3)
print(per_date.to_string())

14 dual-sensor dates: [datetime.date(2026, 4, 11), datetime.date(2026, 5, 2), datetime.date(2026, 5, 3), datetime.date(2026, 5, 9), datetime.date(2026, 5, 16), datetime.date(2026, 5, 17), datetime.date(2026, 5, 18), datetime.date(2026, 5, 19), datetime.date(2026, 5, 20), datetime.date(2026, 5, 23), datetime.date(2026, 5, 24), datetime.date(2026, 5, 27), datetime.date(2026, 5, 28), datetime.date(2026, 5, 31)]


Total matched pairs across all dual-sensor dates: 5964
Total unmatched (single-sensor-only) travelers: 2046

Per-date direction split of the matched pairs:
direction   A_to_B  B_to_A  total  A_to_B_share
date                                           
2026-04-11      41      17     58         0.707
2026-05-02      38      51     89         0.427
2026-05-03     118      96    214         0.551
2026-05-09     171     151    322         0.531
2026-05-16     672     614   1286         0.523
2026-05-17     169     164    333         0.508
2026-05-18     401     402    803         0.499
2026-05-19     384     335    719         0.534
2026-05-20     184     142    326         0.564
2026-05-23     534     506   1040         0.513
2026-05-24     183     136    319         0.574
2026-05-27      51      37     88         0.580
2026-05-28     217     125    342         0.635
2026-05-31      16       9     25         0.640


## Validation: spot-check a few matched pairs

In [14]:
sample = matched_df.sort_values("match_score", ascending=False).head(5)
for _, row in sample.iterrows():
    print(f"{row['date']}  {row['traveler_id_a']} <-> {row['traveler_id_b']}  "
          f"direction={row['direction']}  gap={row['transit_gap_s']:.1f}s  score={row['match_score']:.2f}")

2026-05-31  A-407-8 <-> B-668-5  direction=B_to_A  gap=10.0s  score=1.00
2026-05-20  A-233-1 <-> B-315-14  direction=B_to_A  gap=10.0s  score=1.00
2026-05-18  A-173-3 <-> B-255-32  direction=B_to_A  gap=10.0s  score=1.00
2026-05-28  A-391-3 <-> B-520-2  direction=A_to_B  gap=10.0s  score=1.00
2026-05-28  A-397-21 <-> B-526-7  direction=A_to_B  gap=10.0s  score=1.00


## Write output

Design doc §6 requires the output to carry **both** matched pairs and the
travelers only one sensor ever saw, the latter with a null direction.

### Schema of `DATA/travelers_matched_direction.csv` / `.parquet`

| column | matched row | unmatched row |
|---|---|---|
| `date` | collection date | collection date |
| `match_status` | `matched` | `unmatched_a_only` / `unmatched_b_only` |
| `traveler_id_a` | Stage-C Sensor-A `Traveler_ID` | set for `unmatched_a_only`, else NaN |
| `traveler_id_b` | Stage-C Sensor-B `Traveler_ID` | set for `unmatched_b_only`, else NaN |
| `direction` | `A_to_B` or `B_to_A` | **NaN** — no direction is inferable |
| `transit_gap_s` | seconds between the two boundary anchors; always in `(0, max_transit_s]` | NaN |
| `match_score` | see scale below | NaN |

`match_score = transit_time_score + rssi_weight * rssi_shape_score`, both
terms bounded to [0, 1], so the range is `[min_match_score, 1 + rssi_weight]`
with the calibrated `rssi_weight`. It is a plausibility score, not a
probability. `transit_gap_s` can no longer be negative or absurdly large:
candidates outside `(0, max_transit_s]` score exactly 0 and are rejected.

In [15]:
out_cols = ["date", "match_status", "traveler_id_a", "traveler_id_b",
            "direction", "transit_gap_s", "match_score"]

matched_out = matched_df.copy()
matched_out["match_status"] = "matched"

unmatched_out = unmatched_df.copy()
if len(unmatched_out):
    unmatched_out["match_status"] = unmatched_out["sensor"].map(
        {"A": "unmatched_a_only", "B": "unmatched_b_only"})
    unmatched_out["traveler_id_a"] = unmatched_out["traveler_id"].where(unmatched_out["sensor"] == "A")
    unmatched_out["traveler_id_b"] = unmatched_out["traveler_id"].where(unmatched_out["sensor"] == "B")
    for c in ("direction", "transit_gap_s", "match_score"):
        unmatched_out[c] = np.nan
    unmatched_out = unmatched_out[out_cols]
else:
    unmatched_out = pd.DataFrame(columns=out_cols)

out = pd.concat([matched_out[out_cols], unmatched_out], ignore_index=True)
out = out.sort_values(["date", "match_status"]).reset_index(drop=True)
out.to_csv(f"{DATA_DIR}/travelers_matched_direction.csv", index=False)
out.to_parquet(f"{DATA_DIR}/travelers_matched_direction.parquet", index=False)
print(f"wrote {len(out)} rows to travelers_matched_direction.csv / .parquet")
print(out["match_status"].value_counts().to_string())
print("\ndirection value counts (NaN = unmatched, by construction):")
print(out["direction"].value_counts(dropna=False).to_string())
print("\ntransit_gap_s of matched rows:")
print(out["transit_gap_s"].describe().round(2).to_string())
print(f"negative transit_gap_s rows: {(out['transit_gap_s'] < 0).sum()} (must be 0)")

wrote 8010 rows to travelers_matched_direction.csv / .parquet
match_status
matched             5964
unmatched_b_only    1120
unmatched_a_only     926

direction value counts (NaN = unmatched, by construction):
direction
A_to_B    3179
B_to_A    2785
NaN       2046

transit_gap_s of matched rows:
count    5964.00
mean       25.47
std        32.24
min         0.00
25%         7.00
50%        13.00
75%        29.00
max       156.00
negative transit_gap_s rows: 0 (must be 0)
